<a href="https://colab.research.google.com/github/eaedk/Machine-Learning-Tutorials-Updated/blob/main/03_computer_vision/01_image_classification_quest_fashion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👕 Quest · Image classification: teach a computer to recognise clothes

**1989, Bell Labs.** Yann LeCun trains a small neural network, later called **LeNet**, to read the handwritten digits
on postal codes and bank cheques. Its secret is the **convolution**: instead of looking at every pixel separately, the
network slides small learnable filters over the image to detect edges, then shapes, then objects. By the late 1990s,
LeNet was reading a large share of the cheques written in the USA.

**2012, ImageNet competition.** A much bigger convolutional network, **AlexNet**, trained on two gaming GPUs, beats
every hand-crafted method by a huge margin on 1.2 million images and 1 000 classes. The deep learning era begins.

**2017, Zalando.** Researchers complain that handwritten digits (MNIST) are now *too easy*. Zalando publishes
**Fashion-MNIST**: 70 000 small grayscale pictures of clothes, same format, much harder. That's our playground.

**Your mission:** classify 28×28 images into 10 clothing classes. **Metric:** accuracy.

| You will practise | Concept |
|---|---|
| What an image is for a computer | tensors, channels, normalisation |
| Feeding data to a network | `Dataset`, `DataLoader`, batches |
| Building a CNN from scratch | convolutions, pooling, `nn.Module` |
| The training loop | forward, loss, backward, optimizer step |
| Diagnosing models | learning curves, **overfitting**, data augmentation |

🧩 Cells marked `# TODO` are yours. 💡 Solutions are hidden below each exercise.

In [ ]:
# ⚙️ Setup: run me first! Finds the course toolkit `mlquest` (Windows, macOS, Linux and Colab).
import os, sys, subprocess, pathlib
if "google.colab" in sys.modules:  # on Colab: grab the repo (once) and the few missing packages
    if not pathlib.Path("/content/Machine-Learning-Tutorials-Updated").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/eaedk/Machine-Learning-Tutorials-Updated", "/content/Machine-Learning-Tutorials-Updated"], check=True)
    os.chdir("/content/Machine-Learning-Tutorials-Updated")
ROOT = pathlib.Path.cwd()
while not (ROOT / "mlquest").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from mlquest import *  # paths, seed_everything, get_device, quest/checkpoint, Leaderboard...
set_style()
SEED = seed_everything(42)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

DEVICE = get_device()
quest("Teach a computer to recognise clothes", chapter="Track 3 · Computer vision · Image classification", xp=300)

## 1. Load Fashion-MNIST 📥

`torchvision` downloads the dataset once into `data/torchvision/`. The `transform` turns each picture into a
**tensor** (a multi-dimensional array that can live on the GPU) and **normalises** pixels around 0 (neural networks
learn faster with inputs of small, centred values).

In [ ]:
CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
to_tensor = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.286,), (0.353,))])

train_set = datasets.FashionMNIST(DATA_DIR / "torchvision", train=True, download=True, transform=to_tensor)
test_set = datasets.FashionMNIST(DATA_DIR / "torchvision", train=False, download=True, transform=to_tensor)
print(f"train: {len(train_set):,} images · test: {len(test_set):,} images")

show_images([train_set[i][0] for i in range(18)], titles=[CLASSES[train_set[i][1]] for i in range(18)],
            cmap="gray", ncols=9, size=1.5)

### 🎯 Exercise 1: what does an image look like for PyTorch?
Take the first image of `train_set` (a pair `(image, label)`) and store its shape, its smallest and largest value.

In [ ]:
image, label = train_set[0]
shape = ...        # TODO: a tuple, e.g. tuple(image.shape)
pixel_min = ...    # TODO: image.min().item()
pixel_max = ...    # TODO
print(shape, round(pixel_min, 2), round(pixel_max, 2), CLASSES[label])

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
image, label = train_set[0]
shape = tuple(image.shape)
pixel_min = image.min().item()
pixel_max = image.max().item()
print(shape, round(pixel_min, 2), round(pixel_max, 2), CLASSES[label])
```

</details>

In [ ]:
check(shape == (1, 28, 28) and pixel_min < 0 < pixel_max,
      "Right: 1 channel (grayscale) × 28 × 28 pixels, values centred around 0 after normalisation. "
      "A colour image would have 3 channels (red, green, blue).",
      "Expected shape (1, 28, 28) = (channels, height, width).")

## 2. A classic ML baseline 🧱

Before deep learning, let's see how far a "flat" model goes: each image becomes a vector of 784 pixels and a
logistic regression learns one weight per pixel. It has **no idea** that neighbouring pixels are related.

In [ ]:
from sklearn.linear_model import LogisticRegression

def to_numpy(dataset, n):
    X = torch.stack([dataset[i][0] for i in range(n)]).reshape(n, -1).numpy()
    y = np.array([dataset[i][1] for i in range(n)])
    return X, y

X_tr, y_tr = to_numpy(train_set, 10_000)
X_te, y_te = to_numpy(test_set, 10_000)
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # 300 iterations are enough for a baseline
logreg = LogisticRegression(max_iter=300).fit(X_tr, y_tr)

lb = Leaderboard("fashion_mnist", metric="test_accuracy")
lb.add("Logistic regression (pixels)", logreg.score(X_te, y_te))

## 3. Batches and DataLoaders 🚚

Neural networks learn from **mini-batches**: groups of images processed together on the GPU. A `DataLoader` cuts a
dataset into batches, **shuffles** the training data at every epoch (so the model doesn't learn the order), and
feeds them one by one.

### 🎯 Exercise 2: create the two DataLoaders (batch size 128, shuffle only the training set)

In [ ]:
train_loader = ...   # TODO: DataLoader(train_set, batch_size=128, shuffle=True, num_workers=NUM_WORKERS)
test_loader = ...    # TODO: same for test_set, without shuffling, batch_size=512 is fine

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_set, batch_size=512, shuffle=False, num_workers=NUM_WORKERS)
```

</details>

In [ ]:
xb, yb = next(iter(train_loader))
check(tuple(xb.shape) == (128, 1, 28, 28),
      f"One batch = {tuple(xb.shape)}: (batch, channels, height, width).",
      "The first batch should have shape (128, 1, 28, 28).")

## 4. Build a Convolutional Neural Network 🧠

A CNN stacks two kinds of blocks:
- **Feature extractor**: `Conv2d` (learnable 3×3 filters) → `ReLU` (non-linearity) → `MaxPool2d` (halves the size,
  keeps the strongest signals). Early layers detect edges, deeper ones detect shapes like collars or soles.
- **Classifier**: flatten the feature maps and use `Linear` layers to output **10 scores**, one per class (*logits*).

```
 1×28×28 ─conv─▶ 32×28×28 ─pool─▶ 32×14×14 ─conv─▶ 64×14×14 ─pool─▶ 64×7×7 ─flatten─▶ 3136 ─linear─▶ 128 ─linear─▶ 10
```

### 🎯 Exercise 3: complete the CNN following the diagram

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            ...,   # TODO: a Conv2d from 32 to 64 channels (kernel 3, padding 1), a ReLU and a MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(dropout),
            ...,   # TODO: the final Linear layer: 128 → n_classes
        )

    def forward(self, x):
        return self.classifier(self.features(x))

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))
```

</details>

In [ ]:
model = SmallCNN()
out = model(xb[:8])
n_params = sum(p.numel() for p in model.parameters())
check(tuple(out.shape) == (8, 10),
      f"Your CNN outputs 10 scores per image and has {n_params:,} parameters to learn.",
      "For 8 images the output must be (8, 10). Check the channel numbers 32 → 64 and the last Linear layer.")

## 5. The training loop 🔁

Training = repeating 5 steps for every batch:
1. `optimizer.zero_grad()`: forget the gradients of the previous batch,
2. **forward**: `logits = model(x)`,
3. **loss**: how wrong are we? `CrossEntropyLoss` compares the 10 scores with the true class,
4. **backward**: `loss.backward()` computes, for every weight, in which direction to move to reduce the loss,
5. **step**: `optimizer.step()` moves the weights a little in that direction.

One pass over the whole training set is an **epoch**.

### 🎯 Exercise 4: write `train_one_epoch`

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()                       # enables dropout / augmentation-specific behaviour
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        ...                             # TODO: the 5 steps (zero_grad, forward, loss, backward, step)
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n
```

</details>

The evaluation function is given: no gradients (`torch.no_grad()`), and `model.eval()` switches dropout off.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, epochs=5, lr=1e-3):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, loss_fn)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn)
        history.append(dict(epoch=epoch, train_loss=tr_loss, val_loss=va_loss, train_acc=tr_acc, val_acc=va_acc))
        print(f"epoch {epoch:2d} · train acc {tr_acc:.3f} · val acc {va_acc:.3f} · {time.time() - t0:.0f}s")
    return pd.DataFrame(history)


def plot_history(h, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    h.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], marker="o", title="Loss (lower is better)")
    h.plot(x="epoch", y=["train_acc", "val_acc"], ax=axes[1], marker="o", title="Accuracy")
    fig.suptitle(title, fontweight="bold"); plt.tight_layout(); plt.show()

In [ ]:
seed_everything(SEED)
cnn = SmallCNN()
history = fit(cnn, train_loader, test_loader, epochs=4)
plot_history(history, "SmallCNN on 60 000 images")
lb.add("SmallCNN (4 epochs)", history["val_acc"].iloc[-1])
lb.show()

In [ ]:
check(history["val_acc"].iloc[-1] > 0.88,
      "Your CNN beats the pixel-based logistic regression by several points. Convolutions see shapes!",
      "Expected > 88% test accuracy. Check your training loop (did you call optimizer.step()?).")
checkpoint("fashion/cnn", xp=100, msg="You trained your first CNN!")

## 6. 🔥 Overfitting, live

In competitions you rarely have 60 000 labelled images. Let's pretend we only have **1 000**, and train for longer.
Watch the two curves: when the training accuracy keeps rising while the validation accuracy stalls (and the validation
loss goes **up**), the model is **memorising** the training images instead of learning general rules. That's
**overfitting**.

In [ ]:
small_train = Subset(train_set, range(1000))
small_loader = DataLoader(small_train, batch_size=64, shuffle=True, num_workers=NUM_WORKERS)
seed_everything(SEED)
h_overfit = fit(SmallCNN(), small_loader, test_loader, epochs=40)
plot_history(h_overfit, "1 000 images, no regularisation: overfitting")
lb.add("SmallCNN · 1k images", h_overfit["val_acc"].iloc[-1])

### 🎯 Exercise 5: fight overfitting with data augmentation + dropout

**Data augmentation** creates new training images on the fly with small random changes (flip, shift, rotation…) so
the network never sees exactly the same image twice. **Dropout** randomly switches off neurons during training so the
network can't rely on any single one.

Build `augment`: a random horizontal flip, then a small random affine transform (rotation up to 5°, translation up to
5%), then the same `ToTensor` + `Normalize` as before. Then train a `SmallCNN(dropout=0.25)`.

In [ ]:
augment = transforms.Compose([
    ...,   # TODO: transforms.RandomHorizontalFlip()
    ...,   # TODO: transforms.RandomAffine(degrees=5, translate=(0.05, 0.05))
    transforms.ToTensor(), transforms.Normalize((0.286,), (0.353,)),
])
aug_train = Subset(datasets.FashionMNIST(DATA_DIR / "torchvision", train=True, transform=augment), range(1000))
aug_loader = DataLoader(aug_train, batch_size=64, shuffle=True, num_workers=NUM_WORKERS)
seed_everything(SEED)
h_aug = fit(..., aug_loader, test_loader, epochs=40)   # TODO: a SmallCNN with dropout=0.25
plot_history(h_aug, "1 000 images + augmentation + dropout")
lb.add("SmallCNN · 1k images + augmentation", h_aug["val_acc"].iloc[-1])

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
    transforms.ToTensor(), transforms.Normalize((0.286,), (0.353,)),
])
aug_train = Subset(datasets.FashionMNIST(DATA_DIR / "torchvision", train=True, transform=augment), range(1000))
aug_loader = DataLoader(aug_train, batch_size=64, shuffle=True, num_workers=NUM_WORKERS)
seed_everything(SEED)
h_aug = fit(SmallCNN(dropout=0.25), aug_loader, test_loader, epochs=40)
plot_history(h_aug, "1 000 images + augmentation + dropout")
lb.add("SmallCNN · 1k images + augmentation", h_aug["val_acc"].iloc[-1])
```

</details>

In [ ]:
gap_before = h_overfit["train_acc"].iloc[-1] - h_overfit["val_acc"].iloc[-1]
gap_after = h_aug["train_acc"].iloc[-1] - h_aug["val_acc"].iloc[-1]
print(f"Train-validation gap: {gap_before:.1%} without → {gap_after:.1%} with augmentation + dropout")
print(f"Best validation accuracy: {h_overfit['val_acc'].max():.3f} without → {h_aug['val_acc'].max():.3f} with")
check(gap_after < gap_before,
      "The gap shrank: the model stopped memorising. Augmentation is the #1 trick in vision competitions.",
      "Did you pass `augment` to the dataset and dropout=0.25 to the model?")
lb.show()

**Reading the result:** without regularisation, the model reaches ~99% on its 1 000 training images, but that extra
"knowledge" is memorisation: the validation accuracy stopped improving long ago. With augmentation and dropout,
the gap between training and validation shrinks a lot, for about the same best validation score: the model learns
general rules instead of memorising. On tiny 28×28 grayscale images and 1 000 examples, that's all augmentation can
do; on real photos (next notebook) it often brings several points of accuracy.
The real cure for overfitting remains **more data**: compare with the 60 000-image model above!

## 7. Where does the model fail? 🔎

A **confusion matrix** counts, for each true class (rows), what the model predicted (columns). Mistakes are not
random: they tell you what is genuinely hard.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        preds.append(model(x.to(DEVICE)).argmax(1).cpu()); labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()

preds, labels = predict_all(cnn, test_loader)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(labels, preds, display_labels=CLASSES, xticks_rotation=60,
                                        cmap="Blues", colorbar=False, ax=ax)
plt.title("Confusion matrix of the full-data CNN"); plt.show()

wrong = np.where(preds != labels)[0][:12]
show_images([test_set[i][0] for i in wrong], titles=[f"{CLASSES[labels[i]]}→{CLASSES[preds[i]]}" for i in wrong],
            cmap="gray", ncols=6, size=1.9, suptitle="Some mistakes (true → predicted)")

Shirts, T-shirts, pullovers and coats are confused with each other: at 28×28 pixels, even humans hesitate.

## 🏁 Quest complete!

You built and trained a CNN from scratch, saw overfitting with your own eyes and fixed it.

### 🚀 Your turn (bonus XP)
1. Add a third convolution block (64 → 128 channels). Does it help?
2. Replace `Adam` by `SGD(lr=0.05, momentum=0.9)` and compare the curves.
3. Add `nn.BatchNorm2d` after each convolution: training should get faster and more stable.
4. Train on the full 60 000 images with augmentation for 10 epochs: can you reach 93%?

Next: `02_image_classification_app_beans.ipynb`, where we use a network pre-trained on millions of images to
diagnose bean diseases in Uganda.

In [ ]:
checkpoint("fashion/complete", xp=200)
badge("Pixel Whisperer", "👕")
progress()